# core

> Read a Ramabana session log and give a score to the tool calls in it.

In [ ]:
#| default_exp core

In [ ]:
#| export
from dataclasses import dataclass, asdict
from pathlib import Path
import json, sys

from fastcore.script import call_parse

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
import tempfile

## Read the log

Ramabana writes one JSON Lines record for each turn that ends. Each record holds the `prompt`, the
`reply`, a `state`, and an `activity` list. The `activity` list holds each tool call, its arguments,
its result, and a flag that shows if the call was successful.

A turn ends with the state `complete`, `failed`, or `abandoned`. Drona keeps only the turns with the
state `complete`. A turn that a person stopped is not a good example.

In [ ]:
#| export
RAMABANA_HISTORY = Path.home()/'.config/ramabana/agent-history.jsonl'
DONE = ('complete',)

def _row(line):
    try: return json.loads(line) if line.strip() else None
    except json.JSONDecodeError: return None

def report(fn, *args, **kw):
    "Run `fn`. Show an expected failure as a message, not as a traceback."
    try: return fn(*args, **kw)
    except (ValueError, FileNotFoundError) as e:
        print(e, file=sys.stderr)
        sys.exit(2)

def match_session(
    turns,   # turns from `read_history`
    session, # session id, id prefix, or `latest`
):
    "The turns of one session. An exact id has precedence over a prefix."
    ids = [s for t in turns if (s := t.get('session'))]
    if not ids: raise ValueError('no turn carries a session id')
    if session == 'latest': session = ids[-1]
    hits = ([t for t in turns if t.get('session') == session]
            or [t for t in turns if str(t.get('session') or '').startswith(session)])
    if not hits: raise ValueError(f'no session matches {session!r}')
    if len({t.get('session') for t in hits}) > 1: raise ValueError(f'session {session!r} is ambiguous')
    return hits

def read_history(
    path=RAMABANA_HISTORY, # Ramabana `agent-history.jsonl` path
    session=None,          # session id, id prefix, or `latest`; every session when omitted
    states=DONE,           # turn states to keep; `None` keeps every state
):
    "The Ramabana turns, in the sequence that Ramabana wrote them."
    p = Path(path).expanduser()
    if not p.exists(): raise FileNotFoundError(f'no Ramabana history at {p}')
    rows = [t for line in p.read_text().splitlines() if isinstance(t := _row(line), dict)]
    if not rows: raise ValueError(f'{p} has no turns')
    turns = rows if states is None else [t for t in rows if t.get('state', 'complete') in states]
    if not turns:
        raise ValueError(f'no turn in {p.name} has the state {" or ".join(states)}; use --every-state to keep the others')
    return match_session(turns, session) if session else turns

A prefix can identify a session. The name `latest` identifies the newest session.

In [ ]:
tmp = Path(tempfile.mkdtemp())
archive = tmp/'agent-history.jsonl'
rows = [{'session': 'sess-aaa', 'state': 'complete', 'prompt': 'first'},
        {'session': 'sess-bbb', 'state': 'abandoned', 'prompt': 'stopped part way'},
        {'session': 'sess-bbb', 'state': 'complete', 'prompt': 'second'}]
archive.write_text('\n'.join(json.dumps(r) for r in rows) + '\n')

test_eq([t['prompt'] for t in read_history(archive)], ['first', 'second'])
test_eq([t['prompt'] for t in read_history(archive, 'sess-a')], ['first'])
test_eq([t['prompt'] for t in read_history(archive, 'latest')], ['second'])
test_eq(len(read_history(archive, states=None)), 3)

Drona shows an expected failure as a message. It does not show a traceback. There are four such failures: the log file is not present, the prefix identifies more than one session, the prefix identifies no session, and no turn has a session id.

In [ ]:
test_fail(lambda: read_history(tmp/'absent.jsonl'), contains='no Ramabana history at')
test_fail(lambda: read_history(archive, 'sess-'), contains='ambiguous')
test_fail(lambda: read_history(archive, 'nope'), contains='no session matches')

anon = tmp/'anon.jsonl'
anon.write_text(json.dumps({'prompt': 'no session id'}) + '\n')
test_fail(lambda: read_history(anon, 'latest'), contains='no turn carries a session id')

Drona ignores a line that it cannot read, because Ramabana can write the last line while Drona reads the file.

In [ ]:
torn = tmp/'torn.jsonl'
torn.write_text(json.dumps({'session': 's', 'state': 'complete', 'prompt': 'kept'}) + '\n{"session": "s"')
test_eq([t['prompt'] for t in read_history(torn)], ['kept'])

scalar = tmp/'scalar.jsonl'
scalar.write_text('123\n' + json.dumps({'session': 's', 'state': 'complete', 'prompt': 'kept'}) + '\n')
test_eq([t['prompt'] for t in read_history(scalar)], ['kept'])

A log with no usable turn gives a message about the turns. It does not give a message about a session id. An id that is also the prefix of a longer id still identifies its own session.

In [ ]:
empty = tmp/'empty.jsonl'
empty.write_text('')
test_fail(lambda: read_history(empty), contains='has no turns')

stopped = tmp/'stopped.jsonl'
stopped.write_text(json.dumps({'session': 's', 'state': 'abandoned', 'prompt': 'p'}) + '\n')
test_fail(lambda: read_history(stopped), contains='has the state complete')
test_eq(len(read_history(stopped, states=None)), 1)

nested = tmp/'nested.jsonl'
nested.write_text('\n'.join(json.dumps({'session': s, 'state': 'complete', 'prompt': s})
                            for s in ('abc', 'abcd')) + '\n')
test_eq([t['prompt'] for t in read_history(nested, 'abc')], ['abc'])
test_eq([t['prompt'] for t in read_history(nested, 'abcd')], ['abcd'])

## Give a score to the tool calls

Drona examines the tool calls of a turn. Drona does not examine the text of the reply. Three faults
are visible in the log:

- The agent uses a general search tool, but the prompt gives the name of a repository and the name
  of the tool that reads a repository.
- The agent repeats a failed tool call without a change.
- The error text of a failed call shows that the call does not agree with the contract of the tool.

In [ ]:
#| export
@dataclass(frozen=True)
class Finding:
    "One fault in the tool calls of a turn."
    kind: str
    tool: str
    index: int
    message: str

@dataclass(frozen=True)
class Assessment:
    "The score and the findings for one turn or more."
    score: int
    calls: int
    findings: tuple[Finding, ...]

    def dict(self):
        "The assessment as data for JSON."
        return {'score': self.score, 'calls': self.calls, 'findings': [asdict(f) for f in self.findings]}

def _score(findings): return max(0, 100 - 20*len(findings))

In [ ]:
#| export
RESEARCH_TOOLS = {'web_search', 'read_url', 'search_code', 'run_shell'}
PROTOCOL = (
    ('edit_cell', ('could not parse commands',),
     'Use the notebook editor command format from its current tool contract.'),
    ('run_shell', ('usage:', 'unrecognized arguments'),
     'Read the project command contract before retrying.'),
)

def _args_text(action): return ' '.join(str(v) for v in (action.get('args') or {}).values())

def _reads_repo(action):
    return action.get('tool') == 'run_shell' and 'fossick read-gh-repo' in _args_text(action)

def assess_turn(turn):
    "Give a score to the tool calls of one Ramabana turn."
    acts, findings = turn.get('activity') or [], []
    prompt = str(turn.get('prompt') or '').lower()
    if 'github' in prompt and 'fossick' in prompt:
        research = [(i, a) for i, a in enumerate(acts) if a.get('tool') in RESEARCH_TOOLS]
        if research and not _reads_repo(research[0][1]):
            i, a = research[0]
            findings.append(Finding('route', a.get('tool', ''), i,
                                    'Use fossick read-gh-repo as the first repository research call.'))
    seen = set()
    for i, a in enumerate(acts):
        if a.get('ok', True): continue
        tool, detail = a.get('tool', ''), str(a.get('detail') or '').lower()
        key = (tool, json.dumps(a.get('args') or {}, sort_keys=True, default=str))
        if key in seen:
            findings.append(Finding('repeat_failure', tool, i,
                                    'Diagnose or change route before repeating a failed call.'))
        seen.add(key)
        for name, needles, message in PROTOCOL:
            if tool == name and any(n in detail for n in needles):
                findings.append(Finding('tool_protocol', tool, i, message))
                break
    return Assessment(_score(findings), len(acts), tuple(findings))

def assess_history(turns):
    "Give one score to a group of turns."
    each = [assess_turn(t) for t in turns]
    findings = tuple(f for a in each for f in a.findings)
    return Assessment(_score(findings), sum(a.calls for a in each), findings)

The first research call shows if the agent used the correct tool for a repository.

In [ ]:
bad_route = {'prompt': 'Use fossick to research https://github.com/AnswerDotAI/llmdojo',
             'activity': [{'tool': 'web_search', 'ok': True, 'args': {'query': 'llmdojo'}}]}
test_eq([f.kind for f in assess_turn(bad_route).findings], ['route'])

good_route = {'prompt': 'Use fossick to research https://github.com/AnswerDotAI/llmdojo',
              'activity': [{'tool': 'run_shell', 'ok': True,
                            'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'}}]}
test_eq(assess_turn(good_route), Assessment(100, 1, ()))

An incorrect edit that the agent sends two times gives two findings: one for the contract of the tool, and one for the repeated call.

In [ ]:
edit = {'tool': 'edit_cell', 'ok': False, 'args': {'commands': ''},
        'detail': 'could not parse commands: Expecting value'}
test_eq([f.kind for f in assess_turn({'activity': [edit, dict(edit)]}).findings],
        ['tool_protocol', 'repeat_failure', 'tool_protocol'])

A successful call does not decrease the score. A call with no `ok` flag also does not decrease it, because Ramabana uses `ok` for a call that is still in progress.

In [ ]:
test_eq(assess_turn({'activity': [{'tool': 'read_file', 'args': {}}]}), Assessment(100, 1, ()))
test_eq(assess_history([bad_route, good_route]).calls, 2)
test_eq(assess_history([]), Assessment(100, 0, ()))

## Command line

`drona` prints one JSON report. It uses the same log, prefix, and state rules as the rest of Drona.

In [ ]:
#| export
@call_parse
def main(
    history: str=str(RAMABANA_HISTORY), # Ramabana history path
    session: str=None,                  # session id, id prefix, or `latest`
    every_state: bool=False,            # score abandoned and failed turns too
):
    "Give a score to the tool calls in a Ramabana log."
    turns = report(read_history, history, session, states=None if every_state else DONE)
    print(json.dumps(assess_history(turns).dict(), indent=2))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()